In [ ]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 0. configration
# ============================================================
data_file = r'./data/SEM_analysis/All_cohort_final_BrainVital8_mortality.csv'

duration_col = 'time'
event_col    = 'status'
exposure_col = 'BrainVital8'
cohort_col   = 'cohort'
ALL_COVARIATES = ['age', 'sex', 'bmi', 'drink', 'hypertension']

cohort_list = ['ukb', 'HRS', 'KLOSA', 'elsa', 'charls', 'share']

# ============================================================
# 1. Read data & preprocess
# ============================================================
df = pd.read_csv(data_file, low_memory=False)
df[cohort_col] = df[cohort_col].str.strip()

# Unified sex coding: share cohort 1= male,2= female → 1= male,0= female
share_mask = df[cohort_col] == 'share'
df.loc[share_mask, 'sex'] = df.loc[share_mask, 'sex'].map({1: 1, 2: 0})

# Imputation OF MISSING VALUES
for col in ['age', 'sex', 'bmi', 'drink', 'hypertension', exposure_col]:
    if col in df.columns:
        df[col] = df.groupby(cohort_col)[col].transform(lambda x: x.fillna(x.median()))

df = df.dropna(subset=[duration_col, event_col, exposure_col] + ALL_COVARIATES)

print(f"After data preprocessing: {len(df)} ")
print(f"sex distribution:\n{df.groupby(cohort_col)['sex'].value_counts().unstack(fill_value=0)}\n")

# ============================================================
# 2. Auxiliary function
# ============================================================

def make_result_row(label, cohort_name, variable, n, n_event,
                    hr=np.nan, lo=np.nan, hi=np.nan, p=np.nan,
                    conc=np.nan, note=''):
    return {
        'analysis': label, 'cohort': cohort_name, 'variable': variable,
        'n': n, 'n_event': n_event,
        'HR': hr, 'lower95': lo, 'upper95': hi,
        'p': p, 'concordance': conc, 'note': note
    }


def fit_cox_continuous(data, use_covariates, label, cohort_name,
                       strata_col=None, cluster_col_name=None):
    """ Continuous variable Cox regression """
    n = len(data)
    ne = int(data[event_col].sum())
    if n < 20 or ne < 5:
        return [make_result_row(label, cohort_name, 'Continuous', n, ne,
                                note='sample/event too small')]
    try:
        formula = f"{exposure_col} + " + " + ".join(use_covariates)
        cph = CoxPHFitter(penalizer=0.01)
        kw = {'duration_col': duration_col, 'event_col': event_col, 'formula': formula}
        if strata_col:
            kw['strata'] = strata_col
        if cluster_col_name:
            kw['cluster_col'] = cluster_col_name
            kw['robust'] = True
        cph.fit(data, **kw)
        s = cph.summary.loc[exposure_col]
        return [make_result_row(label, cohort_name, 'Continuous', n, ne,
                                round(s['exp(coef)'], 4),
                                round(s['exp(coef) lower 95%'], 4),
                                round(s['exp(coef) upper 95%'], 4),
                                s['p'], round(cph.concordance_index_, 4))]
    except Exception as e:
        return [make_result_row(label, cohort_name, 'Continuous', n, ne,
                                note=str(e)[:120])]


def fit_cox_quartile(data, use_covariates, label, cohort_name,
                     strata_col=None, cluster_col_name=None):
    """
    Quartile Cox: Quartiles were cut by BrainVital8 within each cohort, with Q1 as the reference.   
    For single-cohort analysis: direct qcut.
    For MEGA analysis: qcut separately within cohorts followed by merging.
    """
    n = len(data)
    ne = int(data[event_col].sum())

    if n < 40 or ne < 10:
        return [make_result_row(label, cohort_name, f'Q{q} vs Q1', n, ne,
                                note='sample/event too small') for q in [2, 3, 4]]

    try:
        tmp = data.copy()

        if cohort_name == 'MEGA':
            # MEGA: Quartiles were cut separately within each cohort
            tmp['_Q'] = tmp.groupby(cohort_col)[exposure_col].transform(
                lambda x: pd.qcut(x, q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
            )
        else:
            # Single cohort: direct cut
            tmp['_Q'] = pd.qcut(tmp[exposure_col], q=4,
                                labels=['Q1','Q2','Q3','Q4'], duplicates='drop')

        tmp['_Q'] = tmp['_Q'].astype(str)

        # Check if data are available for all quartiles
        q_counts = tmp['_Q'].value_counts()
        if len(q_counts) < 4:
            return [make_result_row(label, cohort_name, f'Q{q} vs Q1', n, ne,
                                    note=f'only {len(q_counts)} quartile groups') for q in [2, 3, 4]]

        formula = "_Q + " + " + ".join(use_covariates)
        cph = CoxPHFitter(penalizer=0.01)
        kw = {'duration_col': duration_col, 'event_col': event_col, 'formula': formula}
        if strata_col:
            kw['strata'] = strata_col
        if cluster_col_name:
            kw['cluster_col'] = cluster_col_name
            kw['robust'] = True
        cph.fit(tmp, **kw)

        results = []
        for q_label in ['Q2', 'Q3', 'Q4']:
            # The dummy variable name of the t.xxx format
            possible_names = [f"_Q[T.{q_label}]", f"_Q_{q_label}", q_label]
            s = None
            for pn in possible_names:
                if pn in cph.summary.index:
                    s = cph.summary.loc[pn]
                    break
            if s is not None:
                results.append(make_result_row(
                    label, cohort_name, f'{q_label} vs Q1', n, ne,
                    round(s['exp(coef)'], 4),
                    round(s['exp(coef) lower 95%'], 4),
                    round(s['exp(coef) upper 95%'], 4),
                    s['p'], round(cph.concordance_index_, 4)
                ))
            else:
                results.append(make_result_row(
                    label, cohort_name, f'{q_label} vs Q1', n, ne,
                    note='coef not found in summary'))
        return results

    except Exception as e:
        return [make_result_row(label, cohort_name, f'Q{q} vs Q1', n, ne,
                                note=str(e)[:120]) for q in [2, 3, 4]]


# ---------- Composite call function ----------

def run_single_cohort(data, label, cohort_name, exclude_covs=None):
    """Single cohort: continuous + quartile"""
    use_covs = [c for c in ALL_COVARIATES if c not in (exclude_covs or [])]
    results = []
    results += fit_cox_continuous(data, use_covs, label, cohort_name)
    results += fit_cox_quartile(data, use_covs, label, cohort_name)
    return results


def run_mega(data, label, exclude_covs=None):
    """MEGA: consecutive + quartile"""
    use_covs = [c for c in ALL_COVARIATES if c not in (exclude_covs or [])]
    tmp = data.copy()
    tmp['_strata']  = tmp[cohort_col].astype(str)
    tmp['_cluster'] = tmp[cohort_col].astype(str)

    results = []
    results += fit_cox_continuous(tmp, use_covs, label, 'MEGA',
                                  strata_col='_strata', cluster_col_name='_cluster')
    results += fit_cox_quartile(tmp, use_covs, label, 'MEGA',
                                strata_col='_strata', cluster_col_name='_cluster')
    return results


def run_full(data, label, exclude_covs=None):
    """All cohorts + MEGA, continuous + quartile"""
    results = []
    for cname in cohort_list:
        sub = data[data[cohort_col] == cname]
        if len(sub) > 0:
            results += run_single_cohort(sub, label, cname, exclude_covs)
    results += run_mega(data, label, exclude_covs)
    return results


# ============================================================
# 3. Run all analyses
# ============================================================
all_results = []

print(">>> Overall ...")
all_results += run_full(df, label='Overall')

print(">>> Age stratification ...")
all_results += run_full(df[df['age'] >= 65], label='Age >= 65')
all_results += run_full(df[df['age'] < 65],  label='Age < 65')

print(">>> Sex stratification ...")
all_results += run_full(df[df['sex'] == 1], label='Male',   exclude_covs=['sex'])
all_results += run_full(df[df['sex'] == 0], label='Female', exclude_covs=['sex'])

print(">>> BMI stratification ...")
all_results += run_full(df[df['bmi'] >= 28], label='BMI >= 28')
all_results += run_full(df[df['bmi'] < 28],  label='BMI < 28')

print(">>> Exclude FU <= 2yr ...")
all_results += run_full(df[df[duration_col] > 2], label='Exclude FU <= 2yr')

# ============================================================
# 4. Summary export
# ============================================================
result_df = pd.DataFrame(all_results)

result_df['p'] = result_df['p'].apply(
    lambda x: f'{x:.2e}' if pd.notna(x) and x < 0.001 else (f'{x:.4f}' if pd.notna(x) else '')
)
result_df['HR (95% CI)'] = result_df.apply(
    lambda r: f"{r['HR']:.4f} ({r['lower95']:.4f}-{r['upper95']:.4f})"
    if pd.notna(r['HR']) else '', axis=1
)

col_order = ['analysis', 'cohort', 'variable', 'n', 'n_event',
             'HR', 'lower95', 'upper95', 'HR (95% CI)', 'p', 'concordance', 'note']
result_df = result_df[col_order]

output_file = 'cox_brainvital8_summary.csv'
result_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n{'='*60}")
print(f"All done! The results have been saved to: {output_file}")
print(f"Total {len(result_df)}  rows of results.")
print(f"{'='*60}")
print(result_df.to_string(index=False))